# NB_patient_360
Builds `<catalog>.gold.patient_360` per the STTM (`sttm_silver_to_gold.xlsx`, sheet *patient_360*).

* **Grain:** one row per patient
* **Driving table:** `<catalog>.silver.patients`
* **Joins (LEFT):** `silver.hospitals` on `registered_hospital_id = hospital_id`; `silver.insurance_providers` on `insurance_provider_id`; `silver.lab_results` aggregated by `patient_id`
* **Filters:** none
* **Load:** FULL - `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
dbutils.widgets.text("catalog", "adb_caren")
catalog = dbutils.widgets.get("catalog").strip()

source_tables = ["patients", "hospitals", "insurance_providers", "lab_results"]
missing = [t for t in source_tables if not spark.catalog.tableExists(f"{catalog}.silver.{t}")]
if missing:
    raise ValueError(f"Missing source tables in {catalog}.silver: {missing}")

spark.sql("CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.gold')", args={"catalog": catalog})

In [0]:
patient_360_sql = """
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.patient_360') AS
WITH lab_agg AS (
  SELECT patient_id,
         COUNT(*)                             AS total_tests,
         COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
         COUNT_IF(result_status = 'Critical') AS critical_test_count,
         MAX(test_date)                       AS last_test_date
  FROM IDENTIFIER(:catalog || '.silver.lab_results')
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                            AS patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                        AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), p.date_of_birth) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  CAST(p.registered_hospital_id AS INT)                                AS hospital_id,
  h.hospital_name,
  CAST(p.insurance_provider_id AS INT)                                 AS insurance_provider_id,
  i.provider_name                                                      AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                              AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                      AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                      AS critical_test_count,
  l.last_test_date
FROM IDENTIFIER(:catalog || '.silver.patients') p
LEFT JOIN IDENTIFIER(:catalog || '.silver.hospitals') h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN IDENTIFIER(:catalog || '.silver.insurance_providers') i
  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id
"""

spark.sql(patient_360_sql, args={"catalog": catalog})
print(f"Rebuilt {catalog}.gold.patient_360")

In [0]:
gold_df = spark.table(f"{catalog}.gold.patient_360")
source_count = spark.table(f"{catalog}.silver.patients").count()
gold_count = gold_df.count()
distinct_keys = gold_df.select("patient_id").distinct().count()

print(f"silver.patients rows : {source_count}")
print(f"gold.patient_360 rows: {gold_count}")
assert gold_count == source_count, "Row count differs from driving table - LEFT JOIN policy violated"
assert distinct_keys == gold_count, "Duplicate patient_id found - grain violated"

display(gold_df.limit(20))

In [0]:
# Final summary
target_table = f"{catalog}.gold.patient_360"
load_type_normalized = "FULL"

result_df = spark.read.table(target_table)
result_count = result_df.count()
print(f"\n{'='*60}")
print(f"✅ {load_type_normalized} load completed for {target_table}")
print(f"📊 Total rows in target table after load: {result_count}")
print(f"{'='*60}")

dbutils.notebook.exit(str(result_count))